## Creating a filtered view

### Business requirement: The finance department regularly analyzes successful transactions. Create a reusable dataset that excludes failed and pending transactions.

In [ ]:
CREATE VIEW vv_successful_transaction AS
SELECT
    transacion_id,
    customer_id amount,
    transaction_date
FROM payment_transaction
WHERE
    status = 'Success';

## Creating a view with JOINs

### Business requirement: Analysts need transaction records enriched with customer names and countries without repeatedly writing JOIN statements.

In [ ]:
CREATE VIEW vv_customer_transaction AS
SELECT t.transaction_id, t.customer_id, t.amount, t.status, c.customer_name, c.country, t.transaction_date
FROM
    payment_transaction t
    LEFT JOIN customer c ON t.customer_id = c.customer_id;

## Views with CASE WHEN and conditional aggregation

### Business requirement: The operations department wants to monitor transaction success rates and payment statuses for each customer.

In [ ]:
CREATE VIEW vv_customer_payment_metrics AS
SELECT
    customer_id,
    COUNT(*) AS total_transaction,
    SUM(
        CASE
            WHEN status = 'Success' THEN 1
            ELSE 0
        END
    ) AS successful_transactions,
    SUM(
        CASE
            WHEN status = 'Failed' THEN 1
            ELSE 0
        END
    ) AS failed_transactions,
    SUM(
        CASE
            WHEN status = 'Pending' THEN 1
            ELSE 0
        END
    ) AS pending_transactions,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN status = 'Success' THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS success_rate
FROM payment_transaction
GROUP BY
    customer_id;

## Creating a data-cleaning staging view

### Business requirement: Create a standardized customer dataset for downstream transformations.

In [ ]:
CREATE VIEW vv_customer_staging AS
SELECT
    customer_id,
    NULLIF(LOWER(TRIM(email)), '') AS email,
    NULLIF(UPPER(TRIM(country)), '') AS country,
    NULLIF(TRIM(customer_name), '') AS customer_name
FROM raw_customer;

## Creating a view with multiple CTEs

### Business requirement: Calculate the net amount of every successful transaction after deducting refunds, without duplicating transaction records.

In [ ]:
CREATE VIEW vv_net_transactions AS
WITH
    successful_transaction AS (
        SELECT
            transaction_id,
            customer_id,
            amount,
            status,
            transaction_date
        FROM payment_transaction
        WHERE
            status = 'Success'
    )

WITH
    refunded_summary AS (
        transaction_id,
        SUM(refunded_amount) AS total_refunded
        FROM refunded
        GROUP BY
            transaction_id
    )
WITH
    net_summary AS (
        SELECT
            t.transaction_id,
            t.customer_id,
            t.transaction_date t.amount AS original_amount COALESCE(r.total_refunded, 0) AS refunded_amount,
            t.amount - (COALESCE(r.total_refunded, 0)) AS net_amount
        FROM
            successful_transaction t
            LEFT JOIN refunded_summary r ON t.transaction_id = r.transaction_id
    )
SELECT
    transaction_id,
    customer_id,
    transaction_date,
    original_amount,
    refunded_amount net_amount
FROM net_summary
GROUP BY
    transaction_id;